# Dynamic Pricing with Causal Inference

This notebook demonstrates dynamic pricing strategies that adapt to market conditions and customer behavior over time:
- Time-based pricing models
- Market response estimation
- Adaptive pricing algorithms
- Real-time optimization

## Objective
Develop pricing strategies that respond to changing market conditions while maximizing long-term profitability.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

from utils import generate_synthetic_insurance_data, set_style

# Set plotting style
set_style()

print("Libraries imported successfully!")
print("Dynamic Pricing Framework Ready")

## 1. Generate Time-Series Data

Create synthetic data with temporal patterns and market dynamics.

In [ ]:
def generate_dynamic_insurance_data(n_periods=24, customers_per_period=500):
    """Generate time-series insurance data with market dynamics."""
    
    np.random.seed(42)
    all_data = []
    
    for period in range(n_periods):
        # Market conditions that change over time
        market_trend = np.sin(2 * np.pi * period / 12) * 0.2  # Seasonal pattern
        economic_cycle = np.cos(2 * np.pi * period / 24) * 0.15  # Economic cycle
        competition_intensity = 0.1 + 0.05 * np.sin(2 * np.pi * period / 6)  # Competition
        
        # Generate base customer data
        period_data = generate_synthetic_insurance_data(n_samples=customers_per_period)
        
        # Add time-based features
        period_data['period'] = period
        period_data['market_trend'] = market_trend
        period_data['economic_cycle'] = economic_cycle
        period_data['competition_intensity'] = competition_intensity
        
        # Adjust prices based on market conditions
        price_multiplier = 1 + market_trend + economic_cycle + competition_intensity
        period_data['market_price'] = period_data['price'] * price_multiplier
        
        # Adjust conversion based on market conditions and price sensitivity
        price_sensitivity = -0.0005  # Elasticity coefficient
        market_effect = market_trend * 0.3 + economic_cycle * 0.2 - competition_intensity * 0.5
        
        # Recalculate conversion with dynamic effects
        conversion_adjustment = (price_sensitivity * (period_data['market_price'] - period_data['price']) + 
                               market_effect + 
                               np.random.normal(0, 0.05, len(period_data)))
        
        period_data['dynamic_conversion'] = np.clip(
            period_data['conversion'] + conversion_adjustment, 0, 1
        )
        
        # Adjust profit based on new conversion
        cost_per_policy = 300  # Base cost
        period_data['dynamic_profit'] = (
            period_data['dynamic_conversion'] * 
            (period_data['market_price'] - cost_per_policy * (1 + competition_intensity))
        )
        
        # Add lag effects (previous period influence)
        if period > 0:
            period_data['lag_conversion'] = all_data[-1]['dynamic_conversion'].mean()
            period_data['lag_price'] = all_data[-1]['market_price'].mean()
        else:
            period_data['lag_conversion'] = period_data['dynamic_conversion'].mean()
            period_data['lag_price'] = period_data['market_price'].mean()
        
        all_data.append(period_data)
    
    # Combine all periods
    combined_data = pd.concat(all_data, ignore_index=True)
    
    return combined_data

# Generate dynamic dataset
dynamic_df = generate_dynamic_insurance_data(n_periods=24, customers_per_period=500)

print(f"Dynamic dataset shape: {dynamic_df.shape}")
print(f"Time periods: {dynamic_df['period'].nunique()}")
print(f"Customers per period: {len(dynamic_df) // dynamic_df['period'].nunique()}")

# Summary statistics by period
period_summary = dynamic_df.groupby('period').agg({
    'market_price': 'mean',
    'dynamic_conversion': 'mean',
    'dynamic_profit': 'mean',
    'market_trend': 'first',
    'economic_cycle': 'first',
    'competition_intensity': 'first'
}).round(3)

print(f"\nFirst 10 periods summary:")
print(period_summary.head(10))

# Visualize market dynamics
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Market conditions over time
axes[0, 0].plot(period_summary.index, period_summary['market_trend'], 'b-', label='Market Trend')
axes[0, 0].plot(period_summary.index, period_summary['economic_cycle'], 'r-', label='Economic Cycle')
axes[0, 0].plot(period_summary.index, period_summary['competition_intensity'], 'g-', label='Competition')
axes[0, 0].set_xlabel('Period')
axes[0, 0].set_ylabel('Market Condition')
axes[0, 0].set_title('Market Conditions Over Time')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Price evolution
axes[0, 1].plot(period_summary.index, period_summary['market_price'], 'purple', linewidth=2)
axes[0, 1].set_xlabel('Period')
axes[0, 1].set_ylabel('Average Price ($)')
axes[0, 1].set_title('Price Evolution')
axes[0, 1].grid(True, alpha=0.3)

# Conversion rate evolution
axes[1, 0].plot(period_summary.index, period_summary['dynamic_conversion'], 'orange', linewidth=2)
axes[1, 0].set_xlabel('Period')
axes[1, 0].set_ylabel('Conversion Rate')
axes[1, 0].set_title('Conversion Rate Evolution')
axes[1, 0].grid(True, alpha=0.3)

# Profit evolution
axes[1, 1].plot(period_summary.index, period_summary['dynamic_profit'], 'red', linewidth=2)
axes[1, 1].set_xlabel('Period')
axes[1, 1].set_ylabel('Average Profit ($)')
axes[1, 1].set_title('Profit Evolution')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Correlation analysis
print(f"\nCorrelation Analysis:")
correlation_features = ['market_price', 'dynamic_conversion', 'dynamic_profit', 
                       'market_trend', 'economic_cycle', 'competition_intensity']
correlation_matrix = period_summary[correlation_features].corr()
print(correlation_matrix.round(3))

## 2. Dynamic Pricing Model

Build predictive models that account for temporal dependencies.

In [ ]:
class DynamicPricingModel:
    def __init__(self, data):
        self.data = data
        self.conversion_model = None
        self.profit_model = None
        self.market_model = None
        self.scaler = StandardScaler()
        
    def prepare_features(self, include_lags=True):
        """Prepare features for dynamic modeling."""
        
        base_features = ['age', 'income', 'risk_score', 'previous_claims', 
                        'market_price', 'market_trend', 'economic_cycle', 
                        'competition_intensity']
        
        if include_lags:
            base_features.extend(['lag_conversion', 'lag_price'])
        
        # Add time-based features
        self.data['period_sin'] = np.sin(2 * np.pi * self.data['period'] / 12)
        self.data['period_cos'] = np.cos(2 * np.pi * self.data['period'] / 12)
        base_features.extend(['period_sin', 'period_cos'])
        
        # Add interaction terms
        self.data['price_competition'] = self.data['market_price'] * self.data['competition_intensity']
        self.data['price_trend'] = self.data['market_price'] * self.data['market_trend']
        base_features.extend(['price_competition', 'price_trend'])
        
        return base_features
    
    def train_models(self, test_size=0.2):
        """Train dynamic pricing models."""
        
        features = self.prepare_features()
        X = self.data[features]
        y_conversion = self.data['dynamic_conversion']
        y_profit = self.data['dynamic_profit']
        
        # Scale features
        X_scaled = self.scaler.fit_transform(X)
        
        # Time-based split (last periods for testing)
        n_test_periods = int(self.data['period'].nunique() * test_size)
        test_periods = sorted(self.data['period'].unique())[-n_test_periods:]
        
        train_mask = ~self.data['period'].isin(test_periods)
        test_mask = self.data['period'].isin(test_periods)
        
        X_train, X_test = X_scaled[train_mask], X_scaled[test_mask]
        y_conv_train, y_conv_test = y_conversion[train_mask], y_conversion[test_mask]
        y_profit_train, y_profit_test = y_profit[train_mask], y_profit[test_mask]
        
        # Train conversion model
        self.conversion_model = RandomForestRegressor(n_estimators=100, random_state=42)
        self.conversion_model.fit(X_train, y_conv_train)
        
        # Train profit model
        self.profit_model = RandomForestRegressor(n_estimators=100, random_state=42)
        self.profit_model.fit(X_train, y_profit_train)
        
        # Evaluate models
        conv_train_score = self.conversion_model.score(X_train, y_conv_train)
        conv_test_score = self.conversion_model.score(X_test, y_conv_test)
        profit_train_score = self.profit_model.score(X_train, y_profit_train)
        profit_test_score = self.profit_model.score(X_test, y_profit_test)
        
        print(f"Model Performance:")
        print(f"  Conversion - Train R²: {conv_train_score:.4f}, Test R²: {conv_test_score:.4f}")
        print(f"  Profit - Train R²: {profit_train_score:.4f}, Test R²: {profit_test_score:.4f}")
        
        # Feature importance
        feature_importance = pd.DataFrame({
            'feature': features,
            'conversion_importance': self.conversion_model.feature_importances_,
            'profit_importance': self.profit_model.feature_importances_
        }).sort_values('conversion_importance', ascending=False)
        
        print(f"\nTop 10 Most Important Features:")
        print(feature_importance.head(10))
        
        return {
            'features': features,
            'train_periods': sorted(self.data['period'].unique())[:-n_test_periods],
            'test_periods': test_periods,
            'performance': {
                'conversion_train': conv_train_score,
                'conversion_test': conv_test_score,
                'profit_train': profit_train_score,
                'profit_test': profit_test_score
            }
        }
    
    def predict_metrics(self, price, period, market_conditions):
        """Predict conversion and profit for given conditions."""
        
        # Get representative customer data for this period
        period_data = self.data[self.data['period'] == min(period, self.data['period'].max())]
        if len(period_data) == 0:
            # Use last available period
            period_data = self.data[self.data['period'] == self.data['period'].max()]
        
        # Create prediction dataset
        X_pred = period_data[['age', 'income', 'risk_score', 'previous_claims']].copy()
        X_pred['market_price'] = price
        X_pred['market_trend'] = market_conditions['market_trend']
        X_pred['economic_cycle'] = market_conditions['economic_cycle']
        X_pred['competition_intensity'] = market_conditions['competition_intensity']
        
        # Add lag features
        if period > 0:
            prev_period_data = self.data[self.data['period'] == period - 1]
            if len(prev_period_data) > 0:
                X_pred['lag_conversion'] = prev_period_data['dynamic_conversion'].mean()
                X_pred['lag_price'] = prev_period_data['market_price'].mean()
            else:
                X_pred['lag_conversion'] = 0.4  # Default
                X_pred['lag_price'] = 800  # Default
        else:
            X_pred['lag_conversion'] = 0.4
            X_pred['lag_price'] = 800
        
        # Add time features
        X_pred['period_sin'] = np.sin(2 * np.pi * period / 12)
        X_pred['period_cos'] = np.cos(2 * np.pi * period / 12)
        
        # Add interaction terms
        X_pred['price_competition'] = X_pred['market_price'] * X_pred['competition_intensity']
        X_pred['price_trend'] = X_pred['market_price'] * X_pred['market_trend']
        
        # Scale features
        X_pred_scaled = self.scaler.transform(X_pred)
        
        # Make predictions
        conversion_pred = self.conversion_model.predict(X_pred_scaled)
        profit_pred = self.profit_model.predict(X_pred_scaled)
        
        # Aggregate results
        return {
            'avg_conversion': conversion_pred.mean(),
            'total_customers': len(X_pred),
            'expected_conversions': conversion_pred.mean() * len(X_pred),
            'total_profit': profit_pred.sum(),
            'avg_profit_per_customer': profit_pred.mean(),
            'total_revenue': conversion_pred.mean() * len(X_pred) * price
        }

# Initialize and train dynamic pricing model
dynamic_model = DynamicPricingModel(dynamic_df)
training_results = dynamic_model.train_models()

print(f"\nDynamic Pricing Model trained successfully!")
print(f"Training periods: {len(training_results['train_periods'])}")
print(f"Testing periods: {len(training_results['test_periods'])}")
print(f"Total features: {len(training_results['features'])}")

## 3. Adaptive Pricing Algorithm

Implement algorithms that adapt prices based on market feedback.

In [ ]:
class AdaptivePricingAlgorithm:
    def __init__(self, dynamic_model, learning_rate=0.1, exploration_rate=0.1):
        self.model = dynamic_model
        self.learning_rate = learning_rate
        self.exploration_rate = exploration_rate
        self.price_history = []
        self.performance_history = []
        self.market_conditions_history = []
        
    def epsilon_greedy_pricing(self, period, market_conditions, price_bounds=(500, 1500)):
        """Epsilon-greedy pricing strategy with exploration."""
        
        if np.random.random() < self.exploration_rate:
            # Explore: random price within bounds
            price = np.random.uniform(price_bounds[0], price_bounds[1])
            strategy = "explore"
        else:
            # Exploit: optimize based on current model
            price = self.optimize_price(period, market_conditions, price_bounds)
            strategy = "exploit"
        
        return price, strategy
    
    def optimize_price(self, period, market_conditions, price_bounds=(500, 1500)):
        """Optimize price for given period and market conditions."""
        
        def objective(price):
            metrics = self.model.predict_metrics(price[0], period, market_conditions)
            return -metrics['total_profit']  # Negative for minimization
        
        result = minimize(objective, x0=[(price_bounds[0] + price_bounds[1]) / 2], 
                         bounds=[price_bounds], method='L-BFGS-B')
        
        return result.x[0]
    
    def update_strategy(self, period, price, market_conditions, observed_metrics):
        """Update pricing strategy based on observed performance."""
        
        # Record history
        self.price_history.append(price)
        self.performance_history.append(observed_metrics)
        self.market_conditions_history.append(market_conditions)
        
        # Adjust exploration rate based on performance
        if len(self.performance_history) > 1:
            recent_performance = [p['total_profit'] for p in self.performance_history[-5:]]
            if len(recent_performance) > 2:
                # Decrease exploration if performance is improving
                trend = np.polyfit(range(len(recent_performance)), recent_performance, 1)[0]
                if trend > 0:  # Improving
                    self.exploration_rate *= 0.95
                else:  # Declining
                    self.exploration_rate *= 1.05
                
                # Keep exploration rate in reasonable bounds
                self.exploration_rate = np.clip(self.exploration_rate, 0.05, 0.3)
    
    def simulate_adaptive_pricing(self, start_period, n_periods, market_scenario='normal'):
        """Simulate adaptive pricing over multiple periods."""
        
        results = []
        
        for period in range(start_period, start_period + n_periods):
            # Generate market conditions based on scenario
            if market_scenario == 'normal':
                market_conditions = {
                    'market_trend': np.sin(2 * np.pi * period / 12) * 0.2,
                    'economic_cycle': np.cos(2 * np.pi * period / 24) * 0.15,
                    'competition_intensity': 0.1 + 0.05 * np.sin(2 * np.pi * period / 6)
                }
            elif market_scenario == 'crisis':
                market_conditions = {
                    'market_trend': -0.3 + np.sin(2 * np.pi * period / 12) * 0.1,
                    'economic_cycle': -0.2 + np.cos(2 * np.pi * period / 24) * 0.1,
                    'competition_intensity': 0.2 + 0.1 * np.sin(2 * np.pi * period / 6)
                }
            elif market_scenario == 'growth':
                market_conditions = {
                    'market_trend': 0.2 + np.sin(2 * np.pi * period / 12) * 0.1,
                    'economic_cycle': 0.15 + np.cos(2 * np.pi * period / 24) * 0.1,
                    'competition_intensity': 0.05 + 0.03 * np.sin(2 * np.pi * period / 6)
                }
            
            # Determine price using adaptive strategy
            price, strategy = self.epsilon_greedy_pricing(period, market_conditions)
            
            # Predict expected performance
            expected_metrics = self.model.predict_metrics(price, period, market_conditions)
            
            # Simulate observed performance (with noise)
            noise_factor = 0.1
            observed_metrics = {
                'total_profit': expected_metrics['total_profit'] * (1 + np.random.normal(0, noise_factor)),
                'expected_conversions': expected_metrics['expected_conversions'] * (1 + np.random.normal(0, noise_factor)),
                'avg_conversion': expected_metrics['avg_conversion'] * (1 + np.random.normal(0, noise_factor))
            }
            
            # Ensure non-negative values
            for key in observed_metrics:
                observed_metrics[key] = max(0, observed_metrics[key])
            
            # Update strategy based on performance
            self.update_strategy(period, price, market_conditions, observed_metrics)
            
            # Store results
            results.append({
                'period': period,
                'price': price,
                'strategy': strategy,
                'market_trend': market_conditions['market_trend'],
                'economic_cycle': market_conditions['economic_cycle'],
                'competition_intensity': market_conditions['competition_intensity'],
                'expected_profit': expected_metrics['total_profit'],
                'observed_profit': observed_metrics['total_profit'],
                'expected_conversions': expected_metrics['expected_conversions'],
                'observed_conversions': observed_metrics['expected_conversions'],
                'exploration_rate': self.exploration_rate
            })
        
        return pd.DataFrame(results)

# Initialize adaptive pricing algorithm
adaptive_algo = AdaptivePricingAlgorithm(dynamic_model, learning_rate=0.1, exploration_rate=0.15)

print("ADAPTIVE PRICING SIMULATION")
print("=" * 50)

# Simulate different market scenarios
scenarios = ['normal', 'crisis', 'growth']
simulation_results = {}

for scenario in scenarios:
    print(f"\nSimulating {scenario.upper()} market scenario...")
    
    # Reset algorithm for each scenario
    adaptive_algo = AdaptivePricingAlgorithm(dynamic_model, learning_rate=0.1, exploration_rate=0.15)
    
    # Run simulation
    results = adaptive_algo.simulate_adaptive_pricing(start_period=24, n_periods=12, market_scenario=scenario)
    simulation_results[scenario] = results
    
    # Summary statistics
    avg_profit = results['observed_profit'].mean()
    avg_conversions = results['observed_conversions'].mean()
    avg_price = results['price'].mean()
    explore_pct = (results['strategy'] == 'explore').mean() * 100
    
    print(f"  Average profit: ${avg_profit:,.2f}")
    print(f"  Average conversions: {avg_conversions:.0f}")
    print(f"  Average price: ${avg_price:.2f}")
    print(f"  Exploration rate: {explore_pct:.1f}%")

print(f"\nSimulation completed for all scenarios!")

## 4. Performance Analysis

Analyze the performance of different pricing strategies.

In [ ]:
print("ADAPTIVE PRICING PERFORMANCE ANALYSIS")
print("=" * 50)

# Compare with static pricing
def simulate_static_pricing(scenario, static_price=800):
    """Simulate static pricing strategy for comparison."""
    
    static_results = []
    
    for period in range(24, 36):  # Same periods as adaptive
        # Generate market conditions
        if scenario == 'normal':
            market_conditions = {
                'market_trend': np.sin(2 * np.pi * period / 12) * 0.2,
                'economic_cycle': np.cos(2 * np.pi * period / 24) * 0.15,
                'competition_intensity': 0.1 + 0.05 * np.sin(2 * np.pi * period / 6)
            }
        elif scenario == 'crisis':
            market_conditions = {
                'market_trend': -0.3 + np.sin(2 * np.pi * period / 12) * 0.1,
                'economic_cycle': -0.2 + np.cos(2 * np.pi * period / 24) * 0.1,
                'competition_intensity': 0.2 + 0.1 * np.sin(2 * np.pi * period / 6)
            }
        elif scenario == 'growth':
            market_conditions = {
                'market_trend': 0.2 + np.sin(2 * np.pi * period / 12) * 0.1,
                'economic_cycle': 0.15 + np.cos(2 * np.pi * period / 24) * 0.1,
                'competition_intensity': 0.05 + 0.03 * np.sin(2 * np.pi * period / 6)
            }
        
        # Predict performance with static price
        metrics = dynamic_model.predict_metrics(static_price, period, market_conditions)
        
        # Add some noise to simulate reality
        noise_factor = 0.1
        observed_profit = metrics['total_profit'] * (1 + np.random.normal(0, noise_factor))
        observed_conversions = metrics['expected_conversions'] * (1 + np.random.normal(0, noise_factor))
        
        static_results.append({
            'period': period,
            'price': static_price,
            'observed_profit': max(0, observed_profit),
            'observed_conversions': max(0, observed_conversions)
        })
    
    return pd.DataFrame(static_results)

# Compare adaptive vs static pricing
comparison_results = {}

for scenario in scenarios:
    adaptive_results = simulation_results[scenario]
    static_results = simulate_static_pricing(scenario)
    
    # Calculate improvements
    adaptive_avg_profit = adaptive_results['observed_profit'].mean()
    static_avg_profit = static_results['observed_profit'].mean()
    profit_improvement = (adaptive_avg_profit - static_avg_profit) / static_avg_profit
    
    adaptive_avg_conversions = adaptive_results['observed_conversions'].mean()
    static_avg_conversions = static_results['observed_conversions'].mean()
    conversion_improvement = (adaptive_avg_conversions - static_avg_conversions) / static_avg_conversions
    
    comparison_results[scenario] = {
        'adaptive_profit': adaptive_avg_profit,
        'static_profit': static_avg_profit,
        'profit_improvement': profit_improvement,
        'adaptive_conversions': adaptive_avg_conversions,
        'static_conversions': static_avg_conversions,
        'conversion_improvement': conversion_improvement
    }

# Display comparison results
print("ADAPTIVE vs STATIC PRICING COMPARISON:")
for scenario, results in comparison_results.items():
    print(f"\n{scenario.upper()} Market:")
    print(f"  Profit improvement: {results['profit_improvement']:.2%}")
    print(f"  Conversion improvement: {results['conversion_improvement']:.2%}")
    print(f"  Adaptive profit: ${results['adaptive_profit']:,.2f}")
    print(f"  Static profit: ${results['static_profit']:,.2f}")

# Visualize performance comparison
fig, axes = plt.subplots(3, 2, figsize=(15, 18))

for i, scenario in enumerate(scenarios):
    adaptive_data = simulation_results[scenario]
    static_data = simulate_static_pricing(scenario)
    
    # Price evolution
    axes[i, 0].plot(adaptive_data['period'], adaptive_data['price'], 'b-', linewidth=2, label='Adaptive')
    axes[i, 0].plot(static_data['period'], static_data['price'], 'r--', linewidth=2, label='Static')
    axes[i, 0].set_xlabel('Period')
    axes[i, 0].set_ylabel('Price ($)')
    axes[i, 0].set_title(f'{scenario.title()} Market - Price Evolution')
    axes[i, 0].legend()
    axes[i, 0].grid(True, alpha=0.3)
    
    # Profit comparison
    axes[i, 1].plot(adaptive_data['period'], adaptive_data['observed_profit'], 'b-', linewidth=2, label='Adaptive')
    axes[i, 1].plot(static_data['period'], static_data['observed_profit'], 'r--', linewidth=2, label='Static')
    axes[i, 1].set_xlabel('Period')
    axes[i, 1].set_ylabel('Profit ($)')
    axes[i, 1].set_title(f'{scenario.title()} Market - Profit Comparison')
    axes[i, 1].legend()
    axes[i, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Market responsiveness analysis
print(f"\nMARKET RESPONSIVENESS ANALYSIS:")
print("=" * 40)

for scenario in scenarios:
    adaptive_data = simulation_results[scenario]
    
    # Correlation between market conditions and price adjustments
    price_changes = adaptive_data['price'].diff().fillna(0)
    market_correlation = np.corrcoef(adaptive_data['market_trend'], price_changes)[0, 1]
    competition_correlation = np.corrcoef(adaptive_data['competition_intensity'], price_changes)[0, 1]
    
    # Price volatility
    price_volatility = adaptive_data['price'].std()
    
    # Learning effectiveness (profit trend)
    profit_trend = np.polyfit(range(len(adaptive_data)), adaptive_data['observed_profit'], 1)[0]
    
    print(f"\n{scenario.upper()} Market Responsiveness:")
    print(f"  Market trend correlation: {market_correlation:.3f}")
    print(f"  Competition correlation: {competition_correlation:.3f}")
    print(f"  Price volatility: ${price_volatility:.2f}")
    print(f"  Learning trend: ${profit_trend:.2f} per period")

# Summary statistics
print(f"\nOVERALL PERFORMANCE SUMMARY:")
print("=" * 40)

overall_profit_improvement = np.mean([results['profit_improvement'] for results in comparison_results.values()])
overall_conversion_improvement = np.mean([results['conversion_improvement'] for results in comparison_results.values()])

print(f"Average profit improvement across all scenarios: {overall_profit_improvement:.2%}")
print(f"Average conversion improvement across all scenarios: {overall_conversion_improvement:.2%}")

# Best and worst performing scenarios
best_scenario = max(comparison_results.keys(), key=lambda x: comparison_results[x]['profit_improvement'])
worst_scenario = min(comparison_results.keys(), key=lambda x: comparison_results[x]['profit_improvement'])

print(f"\nBest performing scenario: {best_scenario} ({comparison_results[best_scenario]['profit_improvement']:.2%} improvement)")
print(f"Worst performing scenario: {worst_scenario} ({comparison_results[worst_scenario]['profit_improvement']:.2%} improvement)")

print(f"\n" + "="*50)
print("Ready to proceed to Incremental Pricing (Notebook 5)")
print("="*50)